# 用 `create_agent` 打造會自己選工具的 AI：網路搜尋 + 計算機
---

## 這堂課你會學到什麼

1. **Agent 是什麼**：一個「模型 + 工具 + 反覆迴圈」的組合，由模型自己決定下一步要做什麼
2. **工具（Tool）長什麼樣子**：`name`（名字）、`description`（說明）、`args`（輸入欄位）。模型就是靠這三樣東西決定要用哪個工具
3. **`create_agent`**：新版一行就能建立 agent，不再需要 ReAct 的長篇提示詞模板
4. **為什麼答案是 `result["messages"][-1].content`**：agent 回傳的是整段對話紀錄，最終答案在最後一則
5. **Agent 不保證答對**：怎麼從訊息紀錄看出模型哪一步出了問題

## 0. 環境準備

```bash
pip install langchain langchain-community langchain-openai python-dotenv ddgs numexpr
```

| 套件 | 為什麼需要 |
|---|---|
| `langchain` | 提供 `create_agent` |
| `langchain-community` | 提供 `load_tools`、`DuckDuckGoSearchRun` |
| `langchain-openai` | 提供 `ChatOpenAI`（連 OpenAI 的聊天模型） |
| `python-dotenv` | 從 `.env` 檔讀取金鑰 |
| `ddgs` | `DuckDuckGoSearchRun` 真正用來搜尋的套件（原始碼的錯誤訊息是 `pip install -U ddgs`） |
| `numexpr` | 計算機工具（`LLMMathChain`）用它來算數學式 |

## 1. 先看整體：Agent 是什麼？

一般的「提示詞 → 模型 → 答案」是一條直線。Agent 多了一個**迴圈**：模型可以選擇「先去用工具，看到結果後再決定下一步」。

```
        你的問題
           │
           ▼
   ┌───────────────┐   不需要工具    ┌────────────┐
   │ 模型想一想     │ ──────────────► │  最終答案   │
   └───────────────┘                 └────────────┘
           │ 需要工具：模型回傳「我要呼叫哪個工具、參數是什麼」
           ▼
   ┌───────────────┐
   │ 執行工具       │   例：搜尋 / 計算機
   └───────────────┘
           │ 工具的結果被記下來，交還給模型
           └────────────► 回到「模型想一想」
```

重點：**真正執行工具的是程式，不是模型。** 模型只負責說「我想用 `duckduckgo_search`，查詢字串是 …」；`create_agent` 負責實際去執行，再把結果交回給模型。

### 這堂課出現的名詞

| 名詞 | 意思 |
|---|---|
| Agent | 能反覆「想 → 用工具 → 看結果」的 LLM 應用 |
| Tool（工具） | 一個模型可以呼叫的功能，包含名字、說明、輸入欄位 |
| Tool calling | 聊天模型的一種能力：不直接回答文字，而是回傳「要呼叫哪個工具、參數是什麼」的結構化資料 |
| Message（訊息） | 對話裡的一則紀錄：`HumanMessage`（你）、`AIMessage`（模型）、`ToolMessage`（工具結果） |
| `invoke` | 執行一個 LangChain 物件：傳入輸入、取得輸出 |

### 新舊寫法對照（你之前學過的 ReAct 版）

| | 舊：`create_react_agent` + `AgentExecutor` | 新：`create_agent` |
|---|---|---|
| 工具資訊怎麼給模型 | 用 `{tools}`、`{tool_names}` 佔位符寫進提示詞 | 以結構化資料傳給模型的 tool calling 功能 |
| 模型的輸出 | 文字，要程式去解析 `Thought / Action / Observation` | 結構化的 `tool_calls` |
| 提示詞 | 一大段 ReAct 模板 | 一句 `system_prompt` 就夠 |
| 建立步驟 | 兩步（建立 agent，再包進 Executor） | 一步 |
| 對模型的要求 | 一般文字模型即可 | 需要支援 tool calling 的聊天模型 |
| `invoke` 回傳 | `{"input": ..., "output": ...}` | `{"messages": [ ... ]}` |

## 2. 匯入套件

| 匯入 | 用途 |
|---|---|
| `create_agent` | 新版建立 agent 的函式 |
| `load_tools` | 用「名字」載入 LangChain 內建的工具（這裡用 `"llm-math"`） |
| `DuckDuckGoSearchRun` | 網路搜尋工具（DuckDuckGo） |
| `ChatOpenAI` | OpenAI 的聊天模型包裝 |
| `load_dotenv` | 把 `.env` 檔的內容讀進環境變數 |

In [ ]:
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

## 3. 載入金鑰並建立模型

- `MODEL`：模型名稱，單獨放成變數，之後要換模型只改這一行。
- `load_dotenv()`：把 `.env` 裡的 `OPENAI_API_KEY` 讀進環境變數。之後 `ChatOpenAI` 會**自己**去環境變數找金鑰，所以你不用（也不該）把金鑰傳進去。
- `temperature=0`：讓模型「盡量選機率最高的字」，輸出比較穩定。注意是「比較穩定」，**不保證**每次完全相同。

In [ ]:
MODEL = "gpt-3.5-turbo"

# 金鑰放在 .env（OPENAI_API_KEY=...），load_dotenv() 讀進環境變數後，ChatOpenAI 會自己使用。
load_dotenv()

llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
)

## 4. 建立工具

這個 agent 有兩個工具：

1. **`search`**：網路搜尋。`DuckDuckGoSearchRun()` 這行是「從 class 建立一個物件（instance）」。
2. **`llm-math`**：計算機。`load_tools(["llm-math"], llm=llm)` 回傳的是 **list**（可以一次載入很多個工具），所以後面可以用 `tools.append(search)` 把搜尋工具加進同一個 list。

### 為什麼 `llm-math` 要傳 `llm`？

因為這個「計算機」不是普通的計算機：它內部是 `LLMMathChain`，會先**再叫一次模型**，把文字描述的數學問題翻成數學式，才拿去計算。所以它需要一個 LLM。這點在第 7 節會變得很重要。

### 為什麼搜尋工具可以直接放進 `tools`？

因為 `DuckDuckGoSearchRun` **本身就是**一個 LangChain 工具（`BaseTool`），已經有自己的名字、說明與輸入欄位。

> 你可能在舊教學看過 `Tool(name=..., func=search.run, description=...)` 這種包裝，那是 ReAct 時代「把普通函式變成工具」的做法。對 `DuckDuckGoSearchRun` 來說不需要，而且在我測試的版本（`langchain 1.4.6`、`langgraph-prebuilt 1.1.1`）把 `search.run` 這樣包進 `create_agent` 會出現 `NameError: name 'uuid' is not defined`；直接放 `search` 則正常。你的版本是否也會，我不知道。

In [ ]:
#從 DuckDuckGoSearchRun 這個 class 建立一個 search 物件（instance）
#llm-math 本身是一個 LangChain tool
#如果這個工具本身需要使用 LLM 就傳給我建立的llm
search = DuckDuckGoSearchRun()
tools = load_tools(["llm-math"], llm=llm)
tools.append(search)

## 5. 建立 agent

`create_agent` 需要三樣東西：

| 參數 | 意思 |
|---|---|
| `model` | 要用哪個聊天模型（必須支援 tool calling） |
| `tools` | 可以用的工具 list |
| `system_prompt` | 系統提示詞：給模型的整體指示 |

### 為什麼 `system_prompt` 這麼短？

舊的 ReAct 版要自己寫一大段模板，教模型用 `Thought / Action / Observation` 的文字格式思考，還要放 `{tools}`、`{tool_names}`、`{agent_scratchpad}` 佔位符。新版把工具資訊改用結構化方式傳給模型，思考與呼叫流程由 `create_agent` 處理，提示詞只需要放你想交代的事。

### 舊版的兩步，新版合成一步

- 舊：`create_react_agent`（讓 agent「知道有哪些工具可以選」）→ `AgentExecutor`（讓它「真的能執行那些工具」）
- 新：`create_agent` 一次做完

In [ ]:
# 系統提示詞：新版不再需要 {tools}、{tool_names}、{agent_scratchpad} 這些佔位符，
# 也不用自己寫 Thought / Action / Observation 的格式，工具資訊與呼叫流程都由 create_agent 處理。
SYSTEM_PROMPT = "Answer the following question as best you can. "
"Call tools one at a time. Wait for the search result before using the calculator, "
"and never put placeholders in a calculator expression."

# 舊版要分兩步：create_react_agent（讓 Agent「知道有哪些工具可以選」）
# 再包進 AgentExecutor（讓 Executor「真的能執行那些工具」）。
# 新版的 create_agent 一次做完這兩件事。
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

## 6. 提問，並取出答案
### 輸入長什麼樣子？

```python
{"messages": [{"role": "user", "content": question}]}
```

一個 dict，裡面有 `messages` 這個 list，list 裡每個元素是一則訊息（誰說的 `role`、說了什麼 `content`）。現在只有一則：你的問題。

### 為什麼是 `result["messages"][-1].content`？

`invoke` 回傳的**不是一個答案，而是整段對話紀錄**。agent 每做一步，就把一則訊息加進 `messages` 這個 list：

```python
result = {"messages": [
    HumanMessage(...),   # [0] 你的問題
    AIMessage(...),      # [1] 模型說：我要呼叫 duckduckgo_search（tool_calls）
    ToolMessage(...),    # [2] 搜尋工具的結果
    AIMessage(...),      # [3] 模型的最終答案   ← [-1]
]}
```

| 片段 | 意思 |
|---|---|
| `result["messages"]` | 取出整個訊息 list |
| `[-1]` | Python 的「最後一個元素」。agent 最後一步一定是模型給最終答案，所以最後一則就是答案 |
| `.content` | 那則訊息物件裡的文字。訊息物件還有 `tool_calls`、`id`、`usage_metadata` 等欄位，你現在只要文字 |

注意：`result` 是 **dict**，不是 list，所以不能寫 `result[-1]`（會得到 `KeyError: -1`，我實際試過）。要先用 `["messages"]` 取出 list，再用 `[-1]`。

In [ ]:
question = "What is the current price of a Iphone 18 pro is USD? How much would it cost in TWD if the exchange rate is 31.78 NTD for 1 USD."

# question 由呼叫的人傳入；用 return 回傳 agent 的最終答案（字串）。
# 回傳的是 dict，裡面有整串訊息（問題、工具呼叫、工具結果、最終答案），
# 最終答案在最後一則訊息的 .content。
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)
response = result["messages"][-1].content
print(response)

## 8. 重點整理

| # | 重點 |
|---|---|
| 1 | **Agent = 模型 + 工具 + 迴圈。** 模型決定要不要用工具；真正執行工具的是程式 |
| 2 | **工具 = `name` + `description` + `args`。** 模型主要靠 `description` 判斷什麼時候該用哪個工具 |
| 3 | `DuckDuckGoSearchRun` **本身就是工具**，直接放進 `tools`，不需要 `Tool(...)` 包裝 |
| 4 | `load_tools(["llm-math"], llm=llm)` 回傳 **list**；計算機內部會**再叫一次模型**，所以要傳 `llm` |
| 5 | `create_agent(model, tools, system_prompt)` 取代舊的 `create_react_agent` + `AgentExecutor`；不需要 `{tools}` 等佔位符 |
| 6 | 輸入：`{"messages": [{"role": "user", "content": 問題}]}` |
| 7 | 輸出是 **dict**，裡面 `messages` 是整段對話紀錄；最終答案 = `result["messages"][-1].content` |
| 8 | `create_agent` 需要**支援 tool calling** 的聊天模型 |
| 9 | 金鑰放 `.env` + `load_dotenv()`，不要寫進程式碼，`.env` 要加進 `.gitignore` |
| 10 | **答案要檢查。** 看 `tool_calls` 與 `ToolMessage`，確認每個工具拿到的是真實資料，而不是佔位文字 |

### 常見錯誤

| 錯誤 | 原因與處理 |
|---|---|
| `OpenAIError: Missing credentials ...` | 沒有讀到金鑰。確認 `.env` 位置與內容、`load_dotenv()` 在 `ChatOpenAI(...)` **之前**執行 |
| `ImportError: Could not import ddgs python package ...` | 少了 `ddgs`，`pip install -U ddgs`（訊息來自原始碼） |
| `ImportError: LLMMathChain requires the numexpr package ...` | 少了 `numexpr`，`pip install numexpr`（訊息來自原始碼） |
| `KeyError: -1` | 寫成了 `result[-1]`。`result` 是 dict，要寫 `result["messages"][-1]` |
| `NameError: name 'uuid' is not defined` | 在我測試的版本，把 `DuckDuckGoSearchRun().run` 用 `Tool(func=...)` 包起來再放進 `create_agent` 會出現；直接放 `search` 即可 |
| 換成別的模型後不呼叫工具或報錯 | 該模型可能不支援 tool calling。例如本機的 Phi-3 搭配 `create_agent` 的工具呼叫，我**沒有測試過** |

### 動手練習

1. 執行第 7 節的檢查 cell，數數看 `result["messages"]` 有幾則、各是什麼類型。
2. 把問題改成 `"What is 37 * 41?"`，預測模型會不會用搜尋工具，再用檢查 cell 驗證你的預測。
3. 把 `tools.append(search)` 那行註解掉，重新建立 agent 再問 iPhone 的問題，觀察模型怎麼回答。
4. 在 `SYSTEM_PROMPT` 加一句 `"Call tools one at a time. Wait for the search result before using the calculator."`，比較有沒有改善。（**我沒有測試過這個做法有沒有用**）
5. 把 `MODEL` 換成 `"gpt-4o-mini"`，再問同一題，比較答案。（同樣**沒測試過**）
6. 想自訂搜尋工具的說明，不用包裝，直接傳參數：`DuckDuckGoSearchRun(description="Search the web for current information.")`。（我測過這樣可行，`name` 仍是 `duckduckgo_search`）
7. 單獨呼叫計算機：`tools[0].invoke("31.78 * [price in USD]")`，觀察它回傳什麼，檢驗第 7 節的推測。（會消耗 API 呼叫，而且結果不一定每次相同）

### 延伸方向

- 用 `@tool` 裝飾器把自己的 Python 函式變成工具
- 加上記憶：`checkpointer` 與 `thread_id`，讓 agent 記得上一輪對話
- 用 `stream` 一步步看 agent 執行的過程，而不是等全部跑完
- 把 `ChatOpenAI` 換成其他支援 tool calling 的模型